# Pamoka 11 - Model Context Protocol (MCP)

The **Model Context Protocol (MCP)** yra atviras standartas, leidžiantis agentams dinamiškai aptikti ir naudoti įrankius, išteklius ir duomenų šaltinius vykdymo metu. Vietoj to, kad įrankiai būtų įkoduoti į agentą, MCP leidžia agentams prisijungti prie išorinių serverių, kurie pagal poreikį atskleidžia savo galimybes.

Šioje pamokoje sužinosite:
- Kas yra MCP ir kodėl tai svarbu agentų sistemoms
- Kaip veikia MCP kliento-serverio architektūra
- Kaip kurti agentus, kurie naudoja MCP tipo įrankių aptikimą


## Sąranka

**Išankstiniai reikalavimai:**
- Azure AI Foundry projektas su įdiegtu modeliu
- Vykdykite `az login` autentifikacijai


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Kas yra Modelio konteksto protokolas (MCP)?

MCP apibrėžia standartinį būdą, kaip dirbtinio intelekto (DI) agentai gali atrasti ir sąveikauti su išoriniais įrankiais ir duomenų šaltiniais:

- **MCP Server**: Per standartinį protokolą pateikia įrankius, išteklius ir užklausas
- **MCP Client**: Agentų vykdymo aplinka, kuri jungiasi prie serverių ir atranda prieinamas galimybes
- **Dinaminis atradimas**: Agentams nereikia įkoduotų įrankių — jie atranda, kas yra prieinama vykdymo metu

Tai labai naudinga kuriant išplečiamas agentų sistemas, kuriose naujos galimybės gali būti pridėtos nekeičiant agento kodo.


## Kaip veikia MCP

```
┌─────────────┐     discover      ┌─────────────────┐
│  MCP Client  │ ──────────────► │   MCP Server     │
│  (Agent)     │                  │  (Tool Provider) │
│              │ ◄────────────── │                   │
│              │   tool results   │  • list_tools()  │
│              │                  │  • call_tool()   │
└─────────────┘                  │  • resources     │
                                  └─────────────────┘
```

1. Agentas (MCP klientas) prisijungia prie MCP serverio
2. Serveris atsako su prieinamų įrankių ir jų schemų sąrašu
3. Agentas tada gali kviesti bet kurį aptiktą įrankį savo samprotavimo metu
4. Rezultatai grįžta per tą patį protokolą


## MCP įrankių aptikimo simuliavimas

Kadangi tikram MCP serveriui reikalingas veikiantis serverio procesas, mes pademonstruosime šį modelį naudodami `@tool` funkcijas, kurios imituoja tai, ką suteiktų MCP prijungta apgyvendinimo paslauga.

Produkcijoje šie įrankiai būtų dinamiškai aptinkami iš MCP serverio, o ne apibrėžti lokaliai.


In [ ]:
@tool(approval_mode="never_require")
def search_accommodations(
    location: Annotated[str, "The city to search for accommodations"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"] = 2
) -> str:
    """Search for accommodations (simulating an MCP-connected Airbnb tool).
    In production, this would be discovered via MCP from an accommodation service."""
    listings = {
        "Tokyo": [
            {"name": "Shinjuku Modern Apartment", "price": 120, "rating": 4.8},
            {"name": "Traditional Ryokan in Asakusa", "price": 200, "rating": 4.9},
            {"name": "Shibuya Studio", "price": 85, "rating": 4.5},
        ],
        "Paris": [
            {"name": "Le Marais Charming Flat", "price": 150, "rating": 4.7},
            {"name": "Montmartre Artist Loft", "price": 110, "rating": 4.6},
        ],
        "Barcelona": [
            {"name": "Gothic Quarter Penthouse", "price": 130, "rating": 4.8},
            {"name": "Barceloneta Beach Flat", "price": 95, "rating": 4.4},
        ],
    }
    results = listings.get(location, [])
    if not results:
        return f"No accommodations found in {location}"
    output = f"Accommodations in {location} ({check_in} to {check_out}, {guests} guests):\n"
    for listing in results:
        output += f"  - {listing['name']}: ${listing['price']}/night (★{listing['rating']})\n"
    return output


@tool(approval_mode="never_require")
def get_local_experiences(
    location: Annotated[str, "The city to find experiences in"],
    interest: Annotated[str, "Type of experience (food, culture, adventure, etc.)"] = "all"
) -> str:
    """Get local experiences and activities (simulating an MCP-connected tourism tool)."""
    experiences = {
        "Tokyo": {
            "food": ["Tsukiji Market Tour ($45)", "Ramen Making Class ($60)", "Sake Tasting ($35)"],
            "culture": ["Tea Ceremony ($50)", "Samurai Museum ($15)", "Sumo Tournament ($80)"],
            "adventure": ["Mt. Fuji Day Trip ($120)", "Go-kart City Tour ($80)"],
        },
        "Paris": {
            "food": ["Wine & Cheese Tasting ($55)", "Cooking Class ($90)", "Market Tour ($40)"],
            "culture": ["Louvre Guided Tour ($35)", "Montmartre Art Walk ($25)"],
        },
    }
    city_exp = experiences.get(location, {})
    if not city_exp:
        return f"No experiences found in {location}"
    if interest != "all" and interest in city_exp:
        items = city_exp[interest]
        return f"{interest.title()} experiences in {location}:\n" + "\n".join(f"  - {e}" for e in items)
    output = f"All experiences in {location}:\n"
    for cat, items in city_exp.items():
        output += f"\n  {cat.title()}:\n"
        for item in items:
            output += f"    - {item}\n"
    return output

## Agentą kuriant naudojant MCP stiliaus įrankius


In [ ]:
agent = await provider.create_agent(
    tools=[search_accommodations, get_local_experiences],
    name="AccommodationAgent",
    instructions="""You are an accommodation and travel experiences specialist powered by MCP-connected services.

Help travelers find the perfect place to stay and things to do. When searching:
1. Use the search_accommodations tool to find listings
2. Use the get_local_experiences tool to suggest activities
3. Compare options and make personalized recommendations
4. Consider the traveler's budget, interests, and travel style""",
)

response = await agent.run(
    "I'm visiting Tokyo for 5 nights in April with my partner. We love traditional Japanese culture and food. "
    "Find us a place to stay and suggest some experiences.",
    )
print(response)

## MCP gamyboje

Gamybinėje aplinkoje MCP įgalina galingus veikimo šablonus:

- **Dinaminis įrankių aptikimas**: Agentai prisijungia prie MCP serverių ir vykdymo metu aptinka įrankius
- **Atskira architektūra**: Įrankių tiekėjai gali atnaujinti nepriklausomai nuo agento
- **Tarporganizacinis dalijimasis**: Komandos gali per MCP serverius atverti galimybes, kuriomis gali naudotis bet kuris agentas
- **Microsoft Agent Framework palaikymas**: MAF turi įmontuotą MCP kliento palaikymą per `mcp` integraciją

Norėdami naudoti tikrą MCP serverį su MAF, prisijungtumėte per `hosted_mcp_tool()` arba MCP kliento integraciją.

**Sužinokite daugiau:**
- [MCP specifikacija](https://modelcontextprotocol.io/)
- [Microsoft Agent Framework MCP palaikymas](https://github.com/microsoft/agent-framework/tree/main/python/samples/02-agents/mcp)


## Santrauka

Šioje pamokoje sužinojote:
- **MCP** yra atviras standartas dinamiškam įrankių aptikimui tarp agentų ir įrankių tiekėjų
- **kliento-serverio architektūra** leidžia agentams vykdymo metu atrasti galimybes
- MCP leidžia kurti **išplečiamas, atsietas agentų sistemas**, kur įrankiai gali būti pridedami be kodo pakeitimų
- Microsoft Agent Framework teikia **įmontuotą MCP palaikymą** gamybiniam naudojimui


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
Atsakomybės apribojimas:
Šis dokumentas buvo išverstas pasitelkus dirbtinio intelekto vertimo paslaugą Co-op Translator (https://github.com/Azure/co-op-translator). Nors stengiamės užtikrinti tikslumą, atkreipkite dėmesį, kad automatizuoti vertimai gali turėti klaidų ar netikslumų. Originalus dokumentas gimtąja kalba turi būti laikomas pagrindiniu šaltiniu. Svarbios informacijos atveju rekomenduojamas profesionalus žmogaus vertimas. Mes neatsakome už jokius nesusipratimus ar neteisingas interpretacijas, kilusias dėl šio vertimo naudojimo.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
